In [1]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter
from datasets import load_dataset

# Set plot style for better readability
plt.style.use('seaborn-v0_8-whitegrid')
%matplotlib inline

c:\Users\baaqa\Downloads\Narrative Intelligence\narrative-intelligence\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Load the ARF dataset (synthetic relations subset)
print("Loading dataset... This may take a moment on the first run.")
dataset = load_dataset("Despina/project_gutenberg", "synthetic_relations_in_fiction_books")

# Inspect the structure
print(f"Dataset splits: {dataset.keys()}")
print(f"Number of rows in 'train' split: {len(dataset['train']):,}")

Loading dataset... This may take a moment on the first run.


Dataset splits: dict_keys(['train'])
Number of rows in 'train' split: 95,476


# Dataset Reconnaissance: 
### we need to know exactly what fields ARF gives us: what counts as an entity, what a relationship record looks like, whether there's any temporal signal already, and how dirty the data is.

In [3]:
# Look at a single raw example before assuming anything about structure
sample = dataset["train"][0]
print(type(sample))
for key, value in sample.items():
    print(f"{key!r}: {type(value)} -> {value}")

<class 'dict'>
'book_id': <class 'str'> -> 106
'title': <class 'str'> -> Jungle Tales of Tarzan
'author': <class 'str'> -> Edgar Rice Burroughs
'author_gender': <class 'str'> -> male
'author_birth_year': <class 'str'> -> 1875
'author_death_year': <class 'str'> -> 1950
'release_date': <class 'str'> -> Feb 1, 1994
'pg_subjects': <class 'str'> -> ['Tarzan (Fictitious character) -- Fiction', 'Africa -- Fiction', 'Fantasy fiction', 'Jungles -- Fiction', 'Adventure stories', 'Apes -- Fiction']
'topics': <class 'str'> -> ['fantasy fiction', 'stories', 'adventure stories', 'fiction']
'chunk_id': <class 'str'> -> 0
'chunk': <class 'str'> -> ***  JUNGLE TALES OF TARZAN ***

[Illustration]




Jungle Tales of Tarzan

by Edgar Rice Burroughs




Contents

 CHAPTER I. Tarzan's First Love  CHAPTER II. The Capture of Tarzan  CHAPTER III. The Fight for the Balu  CHAPTER IV. The God of Tarzan  CHAPTER V. Tarzan and the Black Boy  CHAPTER VI. The Witch-Doctor Seeks Vengeance  CHAPTER VII.
'relations': <

In [4]:
print("Features schema:")
print(dataset["train"].features)

Features schema:
{'book_id': Value('string'), 'title': Value('string'), 'author': Value('string'), 'author_gender': Value('string'), 'author_birth_year': Value('string'), 'author_death_year': Value('string'), 'release_date': Value('string'), 'pg_subjects': Value('string'), 'topics': Value('string'), 'chunk_id': Value('string'), 'chunk': Value('string'), 'relations': Value('string')}


In [5]:
df = dataset["train"].to_pandas()
print(df.shape)
df.head(3)

(95476, 12)


,book_id,title,author,author_gender,author_birth_year,author_death_year,release_date,pg_subjects,topics,chunk_id,chunk,relations
0,106,Jungle Tales of Tarzan,Edgar Rice Burroughs,male,1875,1950,"Feb 1, 1994","['Tarzan (Fictitious character) -- Fiction', '...","['fantasy fiction', 'stories', 'adventure stor...",0,*** JUNGLE TALES OF TARZAN ***\r\n\r\n[Illust...,[]
1,106,Jungle Tales of Tarzan,Edgar Rice Burroughs,male,1875,1950,"Feb 1, 1994","['Tarzan (Fictitious character) -- Fiction', '...","['fantasy fiction', 'stories', 'adventure stor...",1,The Witch-Doctor Seeks Vengeance CHAPTER VII....,[]
2,106,Jungle Tales of Tarzan,Edgar Rice Burroughs,male,1875,1950,"Feb 1, 1994","['Tarzan (Fictitious character) -- Fiction', '...","['fantasy fiction', 'stories', 'adventure stor...",10,It is true that Taug was no longer the frolics...,"[{'entity1': 'Taug', 'entity2': 'Tarzan', 'ent..."


In [6]:
# Entity-related columns: which fields identify characters/entities?
print("Columns:", df.columns.tolist())

# Null counts — dirty data means canonicalization work in Week 1
print("\nNull counts:")
print(df.isnull().sum())

# Any duplicate rows?
print(f"\nDuplicate rows: {df.duplicated().sum()}")

Columns: ['book_id', 'title', 'author', 'author_gender', 'author_birth_year', 'author_death_year', 'release_date', 'pg_subjects', 'topics', 'chunk_id', 'chunk', 'relations']

Null counts:
book_id              0
title                1
author               1
author_gender        1
author_birth_year    1
author_death_year    1
release_date         1
pg_subjects          1
topics               1
chunk_id             1
chunk                1
relations            1
dtype: int64

Duplicate rows: 0


In [7]:
# 1. Does a relationship type / label column exist, and what values does it take?
# (replace 'relation' with whatever the actual column is called once you see Cell 2/3 output)
if "relation" in df.columns:
    print(df["relation"].value_counts())

# 2. Is there ANY temporal/ordering signal already in the data?
#    e.g. chapter number, sentence position, book order, timestamps
temporal_candidates = [c for c in df.columns if any(
    kw in c.lower() for kw in ["time", "order", "chapter", "position", "sequence", "date"]
)]
print("Possible temporal columns:", temporal_candidates)

Possible temporal columns: ['release_date']


In [8]:
import ast

row = df.iloc[2]
relations = ast.literal_eval(row["relations"])
print(f"Number of relations in this chunk: {len(relations)}")
for r in relations:
    print(r)

Number of relations in this chunk: 2
{'entity1': 'Taug', 'entity2': 'Tarzan', 'entity1Type': 'PER', 'entity2Type': 'PER', 'relation': 'companion_of'}
{'entity1': 'Taug', 'entity2': 'Teeka', 'entity1Type': 'PER', 'entity2Type': 'PER', 'relation': 'companion_of'}


In [9]:
book_106 = df[df["book_id"] == "106"].copy()
book_106["chunk_id"] = book_106["chunk_id"].astype(int)
book_106 = book_106.sort_values("chunk_id")
print(f"Chunks for book 106: {len(book_106)}")
print(f"chunk_id range: {book_106['chunk_id'].min()} to {book_106['chunk_id'].max()}")
print(f"Is it contiguous (no gaps)? {book_106['chunk_id'].tolist() == list(range(book_106['chunk_id'].min(), book_106['chunk_id'].max()+1))}")

Chunks for book 106: 883
chunk_id range: 0 to 882
Is it contiguous (no gaps)? True


In [10]:
import ast

def safe_parse_relations(raw: object) -> list | None:
    """Parse a stringified relations list.

    Returns [] for genuinely empty relations, a list of dicts for valid data,
    or None if the string is malformed (can't be parsed at all).
    """
    if not isinstance(raw, str):
        return None  # covers NaN / non-string cells
    try:
        return ast.literal_eval(raw)
    except (ValueError, SyntaxError):
        return None

df["relations_parsed"] = df["relations"].apply(safe_parse_relations)

malformed = df[df["relations_parsed"].isnull()]
print(f"Malformed/unparseable rows: {len(malformed)} / {len(df):,}")
print(malformed[["book_id", "chunk_id", "relations"]])

Malformed/unparseable rows: 1 / 95,476
               book_id chunk_id relations
95475  ompanion_of'}]"      NaN       NaN


In [11]:
valid = df[df["relations_parsed"].notnull()].copy()
valid["num_relations"] = valid["relations_parsed"].apply(len)

print(f"Valid rows: {len(valid):,} / {len(df):,}")
print(f"Chunks with >=1 relation: {(valid['num_relations'] > 0).sum():,}")
print(f"Total relation instances: {valid['num_relations'].sum():,}")
print(f"Unique books: {valid['book_id'].nunique()}")

Valid rows: 95,475 / 95,476
Chunks with >=1 relation: 60,245
Total relation instances: 128,331
Unique books: 96


In [12]:
def is_contiguous(group: pd.DataFrame) -> bool:
    ids = sorted(group["chunk_id"].astype(int))
    return ids == list(range(ids[0], ids[-1] + 1))

# sample 20 random books, not just book 106 — one example isn't proof
sample_books = valid["book_id"].dropna().unique()
import random
random.seed(42)
sample = random.sample(list(sample_books), min(20, len(sample_books)))

results = {
    bid: is_contiguous(valid[valid["book_id"] == bid])
    for bid in sample
}
print(results)
print(f"\nContiguous in {sum(results.values())}/{len(results)} sampled books")

{'73548': True, '21299': True, '12807': True, '9909': True, '36684': True, '34025': True, '32543': True, '23060': True, '18873': True, '77': True, '619': True, '16630': True, '6941': True, '5111': False, '1329': True, '31858': True, '3322': True, '5658': True, '70653': True, '64264': True}

Contiguous in 19/20 sampled books


In [13]:
book_5111 = valid[valid["book_id"] == "5111"].copy()
book_5111["chunk_id"] = book_5111["chunk_id"].astype(int)
book_5111 = book_5111.sort_values("chunk_id")

ids = book_5111["chunk_id"].tolist()
gaps = [(ids[i], ids[i+1]) for i in range(len(ids)-1) if ids[i+1] - ids[i] > 1]

print(f"Total chunks: {len(ids)}, range {ids[0]}–{ids[-1]}")
print(f"Gaps found: {gaps}")

Total chunks: 506, range 0–506
Gaps found: [(50, 52)]


In [14]:
# Save the cleaned relations dataframe for use in Week 1 graph-building work,
# so later notebooks/scripts don't need to re-download + re-parse from scratch.
valid.to_parquet("../data/arf_chunks_parsed.parquet", index=False)
print("Saved cleaned dataset to data/arf_chunks_parsed.parquet")

Saved cleaned dataset to data/arf_chunks_parsed.parquet


### finding out frequency of aliasing

In [15]:
def extract_entity_names(relations_list: list[dict]) -> set[str]:
    """Pull every entity1/entity2 name out of a chunk's parsed relations."""
    names = set()
    for r in relations_list:
        names.add(r["entity1"])
        names.add(r["entity2"])
    return names

# Pick book 106 (Jungle Tales of Tarzan) since we already know it well
book_id = "106"
book_rows = valid[valid["book_id"] == book_id]

all_names = set()
for relations_list in book_rows["relations_parsed"]:
    all_names |= extract_entity_names(relations_list)

print(f"Unique entity strings in book {book_id}: {len(all_names)}")
for name in sorted(all_names):
    print(name)

Unique entity strings in book 106: 279
Apes
Balu
Bara
Belgian Congo
Bolgani
Bolgani, the gorilla
Bukawai
Bulabantu
Buto
Chamston-Hedding
Dango
Dango, the hyena
Death
Devil-god
English forbears
English lady
English lord
English nobleman
GOD
Gazan
Go-bu-balu
God
God of the Jungle
Gomangani
Goro
Gozan
Gunto
He
Helen of Troy
His
Histah
Horta
Horta, the boar
Ibeto
Ibeto's son
John Clayton
Kala
Kamma
Kerchak
Kudu
Kulonga
Leopold's domain
Little Go-bu-balu
Lord Greystoke
MAN
Mamka
Mangani
Manu
Mbonga
Mbonga's black warriors
Mbonga's people
Mbonga's warriors
Momaya
Momaya's husband
Mumga
My child
Numa
Numa, the lion
Numgo
Pacco
Pamba
Rabba Kega
Sabor
She
Sheeta
Sheeta's mate
Tantor
Tantor's enemies
Tantor, the elephant
Tarzan
Tarzan of the Apes
Taug
Taug's little balu
Teeka
Teeka's balu
Teeka's little one
Thaka
The apes
Tibo
Toog
Toog's tribe
Tublat
Tubuto
Wappi
Young Lord Greystoke
a young she
all-powerful
ancestor
antagonist
ape
ape-boy
ape-man
apes
apes of Kerchak
baby
balu
black boy
black 

In [16]:
from difflib import SequenceMatcher

def similar(a: str, b: str, threshold: float = 0.6) -> bool:
    """Cheap fuzzy-match check, for diagnostic purposes only."""
    return SequenceMatcher(None, a.lower(), b.lower()).ratio() > threshold

names_list = sorted(all_names)
near_duplicates = []
for i in range(len(names_list)):
    for j in range(i + 1, len(names_list)):
        if similar(names_list[i], names_list[j]):
            near_duplicates.append((names_list[i], names_list[j]))

print(f"Candidate near-duplicate pairs: {len(near_duplicates)}")
for pair in near_duplicates:
    print(pair)

Candidate near-duplicate pairs: 495
('Apes', 'The apes')
('Apes', 'ape')
('Apes', 'apes')
('Apes', 'the Apes')
('Balu', 'balu')
('Balu', 'her balu')
('Bara', 'zebra')
('Bolgani', 'Gomangani')
('Bolgani', 'Mbonga')
('Buto', 'Gunto')
('Buto', 'Ibeto')
('Buto', 'Tubuto')
('Dango, the hyena', 'Tantor, the elephant')
('Dango, the hyena', 'the hyenas')
('Death', 'dead father')
('Devil-god', 'devils')
('Devil-god', 'white devil-god')
('English forbears', 'English lady')
('English forbears', 'English lord')
('English forbears', 'English nobleman')
('English lady', 'English lord')
('English lady', 'English nobleman')
('English lord', 'English nobleman')
('GOD', 'God')
('GOD', 'gods')
('Gazan', 'Gozan')
('Gazan', 'Tarzan')
('Go-bu-balu', 'Little Go-bu-balu')
('God', 'gods')
('God of the Jungle', 'the jungle')
('God of the Jungle', 'the terrible white god of the jungle')
('Gomangani', 'Mangani')
('Gomangani', 'she-Gomangani')
('Gomangani', 'the Gomangani')
('Goro', 'grove')
('He', 'She')
('He', '

In [17]:
import re

def normalize_entity_name(name: str) -> str:
    """Normalize an entity name for use as a graph node identifier.

    Lowercases, strips whitespace, and removes a trailing appositive
    descriptor clause (e.g. "Bolgani, the gorilla" -> "bolgani").

    Known limitation: does not resolve generic/collective entities
    (e.g. "apes" vs "the apes") to a canonical form, and does not
    perform cross-alias resolution (e.g. nicknames, titles).
    """
    name = name.strip().lower()
    name = re.sub(r",\s*the\s+.+$", "", name)
    return name.strip()

for name in sorted(all_names):
    normalized = normalize_entity_name(name)
    if normalized != name.strip().lower():
        print(f"{name!r} -> {normalized!r}")

'Bolgani, the gorilla' -> 'bolgani'
'Dango, the hyena' -> 'dango'
'Horta, the boar' -> 'horta'
'Numa, the lion' -> 'numa'
'Tantor, the elephant' -> 'tantor'


In [18]:
from collections import defaultdict

groups = defaultdict(list)
for name in sorted(all_names):
    groups[normalize_entity_name(name)].append(name)

# Show only groups where more than one original name collapsed together
collisions = {k: v for k, v in groups.items() if len(v) > 1}
for canonical, originals in collisions.items():
    print(f"{canonical!r} <- {originals}")

'apes' <- ['Apes', 'apes']
'balu' <- ['Balu', 'balu']
'bolgani' <- ['Bolgani', 'Bolgani, the gorilla']
'dango' <- ['Dango', 'Dango, the hyena']
'god' <- ['GOD', 'God']
'he' <- ['He', 'he']
'his' <- ['His', 'his']
'horta' <- ['Horta', 'Horta, the boar']
'man' <- ['MAN', 'man']
'numa' <- ['Numa', 'Numa, the lion']
'she' <- ['She', 'she']
'tantor' <- ['Tantor', 'Tantor, the elephant']
'the apes' <- ['The apes', 'the Apes']
'young lord greystoke' <- ['Young Lord Greystoke', 'young Lord Greystoke']


### figuring out type of relations and their symmetry to decide upon the type of knowledge graph

In [19]:
# Check: do the same two entities have multiple relation instances
# (same or different relation types) within a book?
from collections import Counter

pair_counts = Counter()
for _, row in valid[valid["book_id"] == "106"].iterrows():
    for r in row["relations_parsed"]:
        pair = tuple(sorted([r["entity1"], r["entity2"]]))
        pair_counts[pair] += 1

repeated_pairs = {pair: count for pair, count in pair_counts.items() if count > 1}
print(f"Entity pairs with >1 relation instance: {len(repeated_pairs)} / {len(pair_counts)}")
for pair, count in sorted(repeated_pairs.items(), key=lambda x: -x[1])[:10]:
    print(pair, count)

Entity pairs with >1 relation instance: 141 / 421
('Tarzan', 'Taug') 90
('Tarzan', 'Teeka') 85
('Taug', 'Teeka') 42
('Momaya', 'Tibo') 38
('Numa', 'Tarzan') 33
('Tantor', 'Tarzan') 29
('Gazan', 'Teeka') 27
('Mbonga', 'Tarzan') 21
('Kala', 'Tarzan') 19
('Sheeta', 'Tarzan') 19


In [20]:
tarzan_taug = [
    r for _, row in valid[valid["book_id"] == "106"].iterrows()
    for r in row["relations_parsed"]
    if {r["entity1"], r["entity2"]} == {"Tarzan", "Taug"}
]
from collections import Counter
print(Counter(r["relation"] for r in tarzan_taug))

Counter({'companion_of': 47, 'rival_of': 17, 'protector_of': 8, 'friend_of': 7, 'enemy_of': 5, 'relative_of': 2, 'mentor_of': 1, 'sacrifices_for': 1, 'leader_of': 1, 'sibling_of': 1})


In [21]:
import sys
from pathlib import Path

project_root = Path.cwd().parent  # assumes notebook is in notebooks/, project root is one level up
sys.path.insert(0, str(project_root))

print(f"Added to sys.path: {project_root}")

Added to sys.path: c:\Users\baaqa\Downloads\Narrative Intelligence\narrative-intelligence


In [22]:
from graph.build_graph import build_book_graph

book_106_rows = valid[valid["book_id"] == "106"]
graph, stats = build_book_graph(book_106_rows, book_id="106")

print(stats)
print(f"\nSample nodes: {list(graph.nodes(data=True))[:3]}")
print(f"\nSample edges: {list(graph.edges(data=True))[:3]}")

# Sanity checks against what we already know from Day 1/2 exploration
print(f"\n'tarzan' in graph: {'tarzan' in graph.nodes}")
print(f"Tarzan-Taug edge count: {graph.number_of_edges('tarzan', 'taug')}")
print(f"Surface forms for 'bolgani': {graph.nodes['bolgani']['surface_forms']}")

GraphBuildStats(book_id='106', num_nodes=265, num_edges=1280, num_chunks_processed=883)

Sample nodes: [('taug', {'entity_type': 'PER', 'surface_forms': {'Taug'}}), ('tarzan', {'entity_type': 'PER', 'surface_forms': {'Tarzan'}}), ('teeka', {'entity_type': 'PER', 'surface_forms': {'Teeka'}})]

Sample edges: [('taug', 'tarzan', {'relation': 'companion_of', 'chunk_id': 10}), ('taug', 'tarzan', {'relation': 'companion_of', 'chunk_id': 153}), ('taug', 'tarzan', {'relation': 'protector_of', 'chunk_id': 155})]

'tarzan' in graph: True
Tarzan-Taug edge count: 64
Surface forms for 'bolgani': {'Bolgani, the gorilla', 'Bolgani'}


In [23]:
both_directions = (
    graph.number_of_edges('tarzan', 'taug') +
    graph.number_of_edges('taug', 'tarzan')
)
print(f"Tarzan-Taug edges (both directions): {both_directions}")

Tarzan-Taug edges (both directions): 90


In [24]:
# For companion_of specifically, does direction look meaningful or arbitrary?
companion_edges = [
    (u, v, d) for u, v, d in graph.edges(data=True)
    if d["relation"] == "companion_of" and {u, v} == {"taug", "tarzan"}
]
for e in companion_edges:
    print(e)

('taug', 'tarzan', {'relation': 'companion_of', 'chunk_id': 10})
('taug', 'tarzan', {'relation': 'companion_of', 'chunk_id': 153})
('taug', 'tarzan', {'relation': 'companion_of', 'chunk_id': 191})
('taug', 'tarzan', {'relation': 'companion_of', 'chunk_id': 27})
('taug', 'tarzan', {'relation': 'companion_of', 'chunk_id': 555})
('taug', 'tarzan', {'relation': 'companion_of', 'chunk_id': 57})
('taug', 'tarzan', {'relation': 'companion_of', 'chunk_id': 574})
('taug', 'tarzan', {'relation': 'companion_of', 'chunk_id': 707})
('taug', 'tarzan', {'relation': 'companion_of', 'chunk_id': 721})
('taug', 'tarzan', {'relation': 'companion_of', 'chunk_id': 825})
('taug', 'tarzan', {'relation': 'companion_of', 'chunk_id': 831})
('taug', 'tarzan', {'relation': 'companion_of', 'chunk_id': 874})
('taug', 'tarzan', {'relation': 'companion_of', 'chunk_id': 876})
('tarzan', 'taug', {'relation': 'companion_of', 'chunk_id': 167})
('tarzan', 'taug', {'relation': 'companion_of', 'chunk_id': 173})
('tarzan', 't

Known open question: Some relation types (e.g. companion_of) appear bidirectionally for the same entity pair with no consistent direction, suggesting the underlying relation is symmetric but ARF's extraction encodes arbitrary sentence-order direction. Others (e.g. protector_of) appear genuinely asymmetric. Whether to (a) leave as-is, (b) maintain a symmetric-relation-type lookup table and normalize at build time, or (c) handle it at query time in retrieval, is deferred until we're building retrieval and can evaluate against real query behavior.


### testing


In [25]:
import networkx as nx

g = nx.MultiDiGraph()
g.add_node("bolgani", entity_type="PER", surface_forms={"Bolgani", "Bolgani, the gorilla"})

try:
    nx.write_gml(g, "test.gml")
    print("Wrote successfully — let's see what it actually wrote:")
    with open("test.gml") as f:
        print(f.read())
except Exception as e:
    print(f"{type(e).__name__}: {e}")

NetworkXError: {'Bolgani, the gorilla', 'Bolgani'} is not a string


### for all the books

In [26]:
from graph.corpus import build_corpus_graphs, save_corpus, load_corpus

corpus = build_corpus_graphs(valid)
save_corpus(corpus, "../data/graphs/corpus.pkl")

# Reload and verify round-trip integrity
reloaded = load_corpus("../data/graphs/corpus.pkl")
assert reloaded["106"].number_of_edges() == corpus["106"].number_of_edges()
assert reloaded["106"].nodes["bolgani"]["surface_forms"] == corpus["106"].nodes["bolgani"]["surface_forms"]
print("Round-trip verified.")

book=     106  chunks=  883  nodes= 265  edges= 1280
book=   12371  chunks= 1078  nodes=1151  edges= 2067
book=   12753  chunks=  864  nodes= 728  edges= 2652
book=   12807  chunks=  765  nodes= 352  edges= 1404
book=    1329  chunks= 1812  nodes= 264  edges= 1122
book=     134  chunks=  367  nodes= 199  edges=  569
book=   14174  chunks= 2171  nodes= 727  edges= 2824
book=   15284  chunks=   23  nodes=  15  edges=   27
book=    1574  chunks=  435  nodes= 899  edges= 1335
book=    1617  chunks=  666  nodes= 278  edges=  774
book=     165  chunks= 1981  nodes= 445  edges= 1814
book=   16630  chunks= 1198  nodes= 682  edges= 1800
book=    1881  chunks= 1457  nodes= 282  edges=  950
book=   18873  chunks=  995  nodes= 501  edges=  841
book=   21299  chunks= 1700  nodes= 427  edges= 1601
book=   21446  chunks=  373  nodes= 332  edges=  363
book=   22066  chunks= 5031  nodes=2691  edges= 5926
book=   23060  chunks=  142  nodes=  84  edges=  141
book=   24584  chunks=  193  nodes= 107  edges

In [27]:
for book_id, book_graph in corpus.items():
    n, e = book_graph.number_of_nodes(), book_graph.number_of_edges()
    if n == 0 or e == 0:
        print(f"SUSPICIOUS: book {book_id} has {n} nodes, {e} edges")

print(f"\nTotal books: {len(corpus)}")
print(f"Total nodes across corpus: {sum(g.number_of_nodes() for g in corpus.values())}")
print(f"Total edges across corpus: {sum(g.number_of_edges() for g in corpus.values())}")

SUSPICIOUS: book 74763 has 0 nodes, 0 edges

Total books: 96
Total nodes across corpus: 44248
Total edges across corpus: 128331


In [28]:
book_74763_rows = valid[valid["book_id"] == "74763"]
print(f"Number of chunks: {len(book_74763_rows)}")
print(f"Total relations across all its chunks: {book_74763_rows['relations_parsed'].apply(len).sum()}")

# peek at a few chunks
print(book_74763_rows[["chunk_id", "relations_parsed"]].head(10))

Number of chunks: 8
Total relations across all its chunks: 0
      chunk_id relations_parsed
86193        0               []
86194        1               []
86195        2               []
86196        3               []
86197        4               []
86198        5               []
86199        6               []
86200        7               []


Known finding:
- Book 74763 has 0 relations across its 8 chunks -> empty graph.
  Likely a very short work with no extractable named-entity relations.
  Kept in corpus for transparency; will be excluded from retrieval-
  ready book list later.

In [29]:
from collections import Counter

entity_type_counts = Counter()
relation_type_counts = Counter()

for relations_list in valid["relations_parsed"]:
    for r in relations_list:
        entity_type_counts[r["entity1Type"]] += 1
        entity_type_counts[r["entity2Type"]] += 1
        relation_type_counts[r["relation"]] += 1

print("Entity type distribution:")
for etype, count in entity_type_counts.most_common():
    print(f"  {etype}: {count:,}")

print(f"\nUnique relation types seen: {len(relation_type_counts)}")
print("Top 15 relation types:")
for rtype, count in relation_type_counts.most_common(15):
    print(f"  {rtype}: {count:,}")

Entity type distribution:
  PER: 231,313
  LOC: 10,397
  FAC: 4,942
  ORG: 4,713
  OBJ: 2,200
  VEH: 1,073
  CNCP: 902
  WTHR: 431
  EVNT: 414
  TIME: 166
  PER/ORG: 57
  SENT: 48
  FAC/LOC: 2
  LOC/EVNT: 2
  Facility: 1
  Location: 1

Unique relation types seen: 1127
Top 15 relation types:
  companion_of: 37,249
  relative_of: 11,894
  child_of: 7,649
  lover_of: 5,936
  friend_of: 5,903
  sibling_of: 5,628
  spouse_of: 5,579
  enemy_of: 5,370
  rival_of: 4,842
  parent_father_of: 3,816
  parent_mother_of: 2,895
  protector_of: 2,740
  travel_to: 2,588
  leader_of: 2,534
  employer_of: 2,264


In [30]:
relation_counter = Counter()
for relations_list in valid["relations_parsed"]:
    for r in relations_list:
        relation_counter[r["relation"]] += 1

# How many relation types occur only once or twice? (signals typo/drift vs real diversity)
rare = [rt for rt, c in relation_counter.items() if c <= 2]
print(f"Relation types occurring <=2 times: {len(rare)} / {len(relation_counter)}")
print(f"Sample of rare ones: {rare[:30]}")

# How much of total volume do the top 48 (ontology size) account for?
top_48_volume = sum(c for _, c in relation_counter.most_common(48))
total_volume = sum(relation_counter.values())
print(f"\nTop 48 relation types account for {top_48_volume:,} / {total_volume:,} instances ({100*top_48_volume/total_volume:.1f}%)")

Relation types occurring <=2 times: 850 / 1127
Sample of rare ones: ['compares_to', 'learns_about', 'screaming at', 'spared the life of', 'succored', 'help', 'avenges', 'interred_within', 'assent_of', 'guarantees', 'suitable_for', 'introduced_by', 'favorite_of', 'rescued_from', 'refused_to_deliver_to', 'foster_sister_of', 'arrived_in', 'mistaken_for', 'serving', 'distinguishes_under', 'wept like', 'carried off', 'taking from', 'obtained_by', 'interred_in', 'searches_for', 'bail_of', 'lived_with', 'verifies', 'petitioned']

Top 48 relation types account for 125,081 / 128,331 instances (97.5%)


### checks and tests for binning

In [31]:
from temporal.binning import compute_num_bins, assign_narrative_bin

num_bins = compute_num_bins(total_relations=1280)
print(f"N = {num_bins}")

bin_for_100 = assign_narrative_bin(chunk_id=100, min_chunk_id=0, max_chunk_id=882, num_bins=num_bins)
print(f"bin for chunk_id=100: {bin_for_100}")

N = 20
bin for chunk_id=100: 3


In [32]:
# Deliberately test a case where the two formulations would diverge:
# a book whose min_chunk_id isn't 0
test_bin = assign_narrative_bin(chunk_id=105, min_chunk_id=5, max_chunk_id=887, num_bins=20)
print(f"bin (min_chunk_id=5 case): {test_bin}")

# compare: what would it be if we (wrongly) used raw chunk_id, ignoring min_chunk_id offset?
naive_bin = assign_narrative_bin(chunk_id=105, min_chunk_id=0, max_chunk_id=887, num_bins=20)
print(f"bin (naive, ignoring offset): {naive_bin}")

bin (min_chunk_id=5 case): 3
bin (naive, ignoring offset): 3


In [33]:
# A book where min_chunk_id is meaningfully large (e.g. heavy front-of-corpus sampling gap)
correct_bin = assign_narrative_bin(chunk_id=105, min_chunk_id=100, max_chunk_id=982, num_bins=20)
print(f"correct (using min_chunk_id offset): {correct_bin}")

naive_bin = assign_narrative_bin(chunk_id=105, min_chunk_id=0, max_chunk_id=982, num_bins=20)
print(f"naive (ignoring offset, chunk_id=105 treated as near book start): {naive_bin}")

correct (using min_chunk_id offset): 1
naive (ignoring offset, chunk_id=105 treated as near book start): 3


### checking edges and apply binning across the corpus

In [34]:
help(graph.edges)

Help on OutMultiEdgeView in module networkx.classes.reportviews object:

class OutMultiEdgeView(OutEdgeView)
 |  OutMultiEdgeView(G)
 |
 |  A EdgeView class for outward edges of a MultiDiGraph
 |
 |  Method resolution order:
 |      OutMultiEdgeView
 |      OutEdgeView
 |      collections.abc.Set
 |      collections.abc.Mapping
 |      collections.abc.Collection
 |      collections.abc.Sized
 |      collections.abc.Iterable
 |      collections.abc.Container
 |      EdgeViewABC
 |      abc.ABC
 |      builtins.object
 |
 |  Methods defined here:
 |
 |  __call__(self, nbunch=None, data=False, *, default=None, keys=False)
 |      Call self as a function.
 |
 |  __contains__(self, e)
 |
 |  __getitem__(self, e)
 |
 |  __iter__(self)
 |
 |  __len__(self)
 |
 |  data(self, data=True, default=None, nbunch=None, keys=False)
 |      Return a read-only view of edge data.
 |
 |      Parameters
 |      ----------
 |      data : bool or edge attribute key
 |          If ``data=True``, then the data

In [35]:
import networkx as nx
help(nx.MultiDiGraph.edges)

Help on cached_property in module networkx.classes.multidigraph:

<functools.cached_property object>
    An OutMultiEdgeView of the Graph as G.edges or G.edges().

    edges(self, nbunch=None, data=False, keys=False, default=None)

    The OutMultiEdgeView provides set-like operations on the edge-tuples
    as well as edge attribute lookup. When called, it also provides
    an EdgeDataView object which allows control of access to edge
    attributes (but does not provide set-like operations).
    Hence, ``G.edges[u, v, k]['color']`` provides the value of the color
    attribute for the edge from ``u`` to ``v`` with key ``k`` while
    ``for (u, v, k, c) in G.edges(data='color', default='red', keys=True):``
    iterates through all the edges yielding the color attribute with
    default `'red'` if no color attribute exists.

    Edges are returned as tuples with optional data and keys
    in the order (node, neighbor, key, data). If ``keys=True`` is not
    provided, the tuples will jus

In [36]:
# cause python to reload the module so we can see changes without restarting the kernel
import importlib
import temporal.trajectory
importlib.reload(temporal.trajectory)
from temporal.trajectory import build_temporal_index, relationship_trajectory, most_active_pairs_in_bin, weighted_mean_bin, book_trajectory

index = build_temporal_index(graph)
trajectory = relationship_trajectory(graph, "tarzan", "taug", index)
print(trajectory)
print(f"Total counted: {sum(trajectory.values())}")

{1: 17, 2: 10, 3: 0, 4: 17, 5: 3, 6: 1, 7: 0, 8: 0, 9: 0, 10: 0, 11: 0, 12: 0, 13: 4, 14: 2, 15: 0, 16: 7, 17: 10, 18: 0, 19: 10, 20: 9}
Total counted: 90


In [37]:
# Step 1: does the raw nbunch edge pull even find anything?
raw_edges = list(graph.edges(nbunch=["tarzan", "taug"], data=True))
print(f"Raw edges touching either node: {len(raw_edges)}")
print(raw_edges[:3])

# Step 2: does the {u,v} filter keep any of them?
filtered = [(u, v, d) for u, v, d in raw_edges if {u, v} == {"tarzan", "taug"}]
print(f"Filtered to tarzan-taug pair only: {len(filtered)}")
print(filtered[:3])

# Step 3: of those, how many pass the canonical relation check?
from graph.relation_ontology import is_canonical_relation
canonical = [(u, v, d) for u, v, d in filtered if is_canonical_relation(d["relation"])]
print(f"Canonical-only: {len(canonical)}")

# accidently the variable name 'graph' was overwritten in the loop above, so we need to check the original graph object
print("tarzan" in graph.nodes)
print("taug" in graph.nodes)
print(f"Total nodes in graph: {graph.number_of_nodes()}")
print(f"Total edges in graph: {graph.number_of_edges()}")
print(list(graph.nodes)[:10])

Raw edges touching either node: 610
[('tarzan', 'younger apes', {'relation': 'companion_of', 'chunk_id': 101}), ('tarzan', 'tantor', {'relation': 'companion_of', 'chunk_id': 101}), ('tarzan', 'tantor', {'relation': 'companion_of', 'chunk_id': 115})]
Filtered to tarzan-taug pair only: 90
[('tarzan', 'taug', {'relation': 'rival_of', 'chunk_id': 11}), ('tarzan', 'taug', {'relation': 'rival_of', 'chunk_id': 12}), ('tarzan', 'taug', {'relation': 'rival_of', 'chunk_id': 13})]
Canonical-only: 90
True
True
Total nodes in graph: 265
Total edges in graph: 1280
['taug', 'tarzan', 'teeka', 'he', 'the elephant', 'the apes of kerchak', 'younger apes', 'tantor', 'kala', 'tarzan of the apes']


In [38]:
book_106_graph = corpus["106"]
index = build_temporal_index(book_106_graph)
trajectory = relationship_trajectory(book_106_graph, "tarzan", "taug", index)
print(trajectory)
print(f"Total counted: {sum(trajectory.values())}")

{1: 17, 2: 10, 3: 0, 4: 17, 5: 3, 6: 1, 7: 0, 8: 0, 9: 0, 10: 0, 11: 0, 12: 0, 13: 4, 14: 2, 15: 0, 16: 7, 17: 10, 18: 0, 19: 10, 20: 9}
Total counted: 90


In [39]:
# Does the dict correctly behave as "0 where missing" for a consumer?
for b in range(1, index.num_bins + 1):
    print(b, trajectory.get(b, 0))

1 17
2 10
3 0
4 17
5 3
6 1
7 0
8 0
9 0
10 0
11 0
12 0
13 4
14 2
15 0
16 7
17 10
18 0
19 10
20 9


In [40]:
index = build_temporal_index(corpus["106"])
result = most_active_pairs_in_bin(corpus["106"], bin_num=1, index=index, top_n=5)
print(result)

[(('tarzan', 'teeka'), 22), (('tarzan', 'taug'), 17), (('taug', 'teeka'), 10), (('sheeta', 'tarzan'), 6), (('sheeta', 'teeka'), 3)]


### patterns across the corpus

In [41]:
weighted_means = []

for book_id, graph in corpus.items():
    if graph.number_of_edges() == 0:
        continue  # e.g. book 74763

    index = build_temporal_index(graph)
    trajectory = book_trajectory(graph, index)
    wm = weighted_mean_bin(trajectory)

    if wm is not None:
        weighted_means.append((book_id, wm, index.num_bins))

print(f"Books with usable data: {len(weighted_means)} / {len(corpus)}")

# Normalize each weighted mean to a 0-1 scale (since num_bins varies per book),
# so books with different bin counts are comparable
normalized = [wm / n for _, wm, n in weighted_means]

print(f"Min (most early-skewed): {min(normalized):.3f}")
print(f"Max (most late-skewed): {max(normalized):.3f}")
print(f"Mean across corpus: {sum(normalized)/len(normalized):.3f}")
print(f"Books skewing early (< 0.4): {sum(1 for x in normalized if x < 0.4)}")
print(f"Books roughly even (0.4-0.6): {sum(1 for x in normalized if 0.4 <= x <= 0.6)}")
print(f"Books skewing late (> 0.6): {sum(1 for x in normalized if x > 0.6)}")

Books with usable data: 95 / 96
Min (most early-skewed): 0.348
Max (most late-skewed): 0.814
Mean across corpus: 0.531
Books skewing early (< 0.4): 1
Books roughly even (0.4-0.6): 89
Books skewing late (> 0.6): 5


### natural language conversions

In [42]:
used_by_examples = [
    r for relations_list in valid["relations_parsed"]
    for r in relations_list if r["relation"] == "used_by"
][:10]
for r in used_by_examples:
    print(r)

{'entity1': 'the blacks', 'entity2': 'fire', 'entity1Type': 'PER', 'entity2Type': 'OBJ', 'relation': 'used_by'}
{'entity1': 'Flint and Sharp', 'entity2': "lawyers' offices", 'entity1Type': 'ORG', 'entity2Type': 'FAC', 'relation': 'used_by'}
{'entity1': 'Excalibur', 'entity2': 'King Arthur', 'entity1Type': 'OBJ', 'entity2Type': 'PER', 'relation': 'used_by'}
{'entity1': 'Sangreal', 'entity2': 'Sir Galahad', 'entity1Type': 'OBJ', 'entity2Type': 'PER', 'relation': 'used_by'}
{'entity1': 'Sangreal', 'entity2': 'his fellows', 'entity1Type': 'OBJ', 'entity2Type': 'PER', 'relation': 'used_by'}
{'entity1': 'corps of cadets', 'entity2': 'mess hall', 'entity1Type': 'ORG', 'entity2Type': 'FAC', 'relation': 'used_by'}
{'entity1': 'Navy', 'entity2': 'field', 'entity1Type': 'ORG', 'entity2Type': 'FAC', 'relation': 'used_by'}
{'entity1': 'Navy', 'entity2': 'dressing room', 'entity1Type': 'ORG', 'entity2Type': 'FAC', 'relation': 'used_by'}
{'entity1': 'Army nine', 'entity2': 'mess', 'entity1Type': 'ORG

### designing database

In [43]:
from collections import Counter

key_counts = Counter()
for _, row in valid[valid["book_id"] == "106"].iterrows():
    for r in row["relations_parsed"]:
        key = (row["chunk_id"], r["entity1"], r["entity2"], r["relation"])
        key_counts[key] += 1

duplicates = {k: c for k, c in key_counts.items() if c > 1}
print(f"Duplicate (chunk_id, entity1, entity2, relation) combos: {len(duplicates)}")
for k, c in list(duplicates.items())[:5]:
    print(k, c)

Duplicate (chunk_id, entity1, entity2, relation) combos: 0


In [44]:

key_counts = Counter()
for _, row in valid.iterrows():
    book_id = row["book_id"]
    chunk_id = row["chunk_id"]
    for r in row["relations_parsed"]:
        key = (book_id, chunk_id, r["entity1"], r["entity2"], r["relation"])
        key_counts[key] += 1

duplicates = {k: c for k, c in key_counts.items() if c > 1}
print(f"Total unique keys: {len(key_counts):,}")
print(f"Duplicate combos across entire corpus: {len(duplicates)}")
for k, c in list(duplicates.items())[:10]:
    print(k, c)

Total unique keys: 128,190
Duplicate combos across entire corpus: 99
('12753', '234', 'King Pellinore', 'lady', 'companion_of') 2
('12753', '332', 'Sir Lancelot', 'giant', 'enemy_of') 2
('12753', '613', 'Sir Lancelot', 'Sir Tristram', 'companion_of') 2
('12807', '763', 'Dick Prescott', 'Greg Holmes', 'companion_of') 2
('134', '198', 'my uncle', 'me', 'relative_of') 2
('134', '244', 'my', 'husband', 'spouse_of') 2
('134', '280', 'my uncle', 'my', 'relative_of') 2
('134', '354', 'my uncle', 'my child', 'relative_of') 2
('14174', '95', 'his', 'wife', 'spouse_of') 2
('1881', '946', 'Carley', 'her aunt', 'relative_of') 2


In [45]:
count_distribution = Counter(duplicates.values())
print(count_distribution)

Counter({2: 93, 3: 3, 4: 1, 6: 1, 35: 1})


In [46]:
target = max(duplicates, key=duplicates.get)
print(f"The 35x combo: {target}")

book_id, chunk_id, e1, e2, relation = target
row = valid[(valid["book_id"] == book_id) & (valid["chunk_id"].astype(str) == str(chunk_id))]
print(row["chunk"].values[0])
print("\nAll matching relations in this chunk:")
for r in row["relations_parsed"].values[0]:
    if r["entity1"] == e1 and r["entity2"] == e2 and r["relation"] == relation:
        print(r)

The 35x combo: ('31858', '1398', 'his', 'his mother', 'child_of')
She could try her hand on his mother's American destinies, and provide her with amusement and a host of friends.

 He felt all the promptings of natural affection when he was actually face to face with his mother once more, and forgot all his doubts in his intense amusement at her na\xc3\xafve surprise before the comfortable immensity of the San Francisco hotels, and the crowds and automobiles in the streets.

 The next day he took her up to the ranch. For a week she stalked about the country, eight hours out of the twenty-four, expressing interest in nothing, although her eyes always softened at her son's approach; and if she manifested no enthusiasm for his adopted country, at least she barely mentioned the one of his heart. At the end of a week she promptly accepted Isabel's suggestion to transfer herself and her grim disgusted maid to the house on Russian Hill.

All matching relations in this chunk:
{'entity1': 'his'

In [47]:
from embedding.embed_relations import build_embedding_records
book_106_rows = valid[valid["book_id"] == "106"]
records = build_embedding_records(book_106_rows)
print(f"Records: {len(records)}")
print(records[0])
print(records[:3])

Records: 1280
EmbeddingRecord(id='106_10_Taug_Tarzan_companion_of', text='Taug is a companion of Tarzan', book_id='106', chunk_id=10, entity1='Taug', entity2='Tarzan', relation='companion_of')
[EmbeddingRecord(id='106_10_Taug_Tarzan_companion_of', text='Taug is a companion of Tarzan', book_id='106', chunk_id=10, entity1='Taug', entity2='Tarzan', relation='companion_of'), EmbeddingRecord(id='106_10_Taug_Teeka_companion_of', text='Taug is a companion of Teeka', book_id='106', chunk_id=10, entity1='Taug', entity2='Teeka', relation='companion_of'), EmbeddingRecord(id='106_100_he_the elephant_companion_of', text='he is a companion of the elephant', book_id='106', chunk_id=100, entity1='he', entity2='the elephant', relation='companion_of')]


In [49]:
# analyze the time taken for each step of the embedding process
import time
from sentence_transformers import SentenceTransformer
MODEL_NAME = "all-MiniLM-L6-v2"

start = time.time()
records = build_embedding_records(book_106_rows)
print(f"build_embedding_records: {time.time() - start:.2f}s")

start = time.time()
model = SentenceTransformer(MODEL_NAME)
print(f"model load: {time.time() - start:.2f}s")

start = time.time()
texts = [r.text for r in records]
embeddings = model.encode(texts, show_progress_bar=True)
print(f"encode: {time.time() - start:.2f}s")

build_embedding_records: 0.50s


c:\Users\baaqa\Downloads\Narrative Intelligence\narrative-intelligence\.venv\Lib\site-packages\huggingface_hub\file_download.py:141: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\baaqa\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 103/103 [00:00<00:0

model load: 178.44s


Batches: 100%|██████████| 40/40 [00:08<00:00,  4.73it/s]


encode: 8.80s
